# Do Combine tracking measures carry information about NFL play? A prior-weight analysis
**NFL Big Data Bowl 2027, Open Track.** This notebook reproduces every number and figure in the Writeup.

**Question.** For players drafted 2023-25, does what the Combine tracking system measures (position-specific drills at 10 Hz) predict how those players play in NFL regular-season games (player tracking, 2023-25), and how much weight does a Combine-informed prior deserve relative to a player's own early NFL plays?

**Design (decision rules written down in dated notes before the analyses they govern; self-recorded, not registered with a third party, see Writeup).**
1. *Unit of analysis is the player*, not the play. About 60-66 players per position have at least 100 usable snaps, so claims are sized to that.
2. *Combine tracking → NFL linkage.* Combine tracking features (tracked 40-yard-dash speed profile) are joined by `nfl_id` to NFL tracking outcomes: in-game top speed (from `game_tracking_*`), and NGS-style outcomes in `player_play` (DL get-off time, WR route separation, OL pressure allowed).
3. *Value of a Combine prior* is expressed as a **model-implied prior weight** (in independent-play equivalents): an empirical-Bayes prior built from Combine numbers is compared with a position-average prior, evaluated leave-one-player-out on each player's *later* 100 plays.
4. *Sensor vs stopwatch* (exploratory, added after the main analysis): is the tracked speed profile a better prior than the official stopwatch numbers?

Data are used under the competition's CC BY-NC 4.0 terms; nothing is redistributed.

In [ ]:
import os, glob, sys, subprocess
# Locate the competition data (Kaggle) or fall back to a local path.
cands = glob.glob("/kaggle/input/**/combine_results.csv", recursive=True)
DATA = os.path.dirname(cands[0]) if cands else os.environ.get("BDB_DATA", ".")
ON_KAGGLE = os.path.isdir("/kaggle/working")
# Player/play-level intermediates (derived from the competition data) go to temp storage and are NOT published
# with the notebook; only figures and aggregate result tables are copied to the public output folder below.
OUTDIR = os.environ.get("BDB_OUT") or ("/kaggle/temp/work" if ON_KAGGLE else "outputs")
try: os.makedirs(OUTDIR, exist_ok=True)
except OSError: OUTDIR = "/tmp/bdb_work"; os.makedirs(OUTDIR, exist_ok=True)
print("data:", DATA); print("outputs:", OUTDIR)
print(sorted(os.listdir(DATA)))

## 1. Which positions have enough data? (position audit)

In [ ]:
import sys
sys.argv = ["x", DATA]
# --- 03_position_audit.py ---
import sys, pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
D = sys.argv[1]
pd.set_option("display.width", 220, "display.max_columns", 60, "display.max_rows", 120)
cr = pd.read_csv(f"{D}/combine_results.csv", na_values=["NA"])
pos = cr.set_index("nfl_id").combine_position
ct = pd.read_csv(f"{D}/combine_tracking.csv", usecols=["event_id","nfl_id","entity_type","drill_type","drill_name","attempt","time","x","y"])
pp = pd.read_csv(f"{D}/player_play.csv", low_memory=False)
g = pd.read_csv(f"{D}/games.csv")
pp = pp.merge(g[["game_id","season_type"]], on="game_id")
pp = pp[pp.season_type=="REG"].copy()
pp["pos"] = pp.nfl_id.map(pos)
pp["is_pass"] = pp.pass_result.notna() & (pp.pass_result.astype(str)!="")
cols = ["route_ran","separation_at_pass_forward","cushion","target","expected_yards_after_catch","yards_after_catch","rec_yards","player_get_off","sack","time_to_qb_hurry","time_to_pressure","pressure_allowed","coverage_assignment","tackle","tackle_for_loss","blitzing"]
for P in ["DL","WR","DB","TE"]:
    d = pp[pp.pos==P]; dp = d[d.is_pass]
    print(f"\n===== {P}: REG rows {len(d)}, players {d.nfl_id.nunique()} of {(pos==P).sum()}; pass-play rows {len(dp)}")
    print("non-null % within pass-play rows:", {c: round(dp[c].notna().mean()*100) for c in cols if dp[c].notna().mean()>0.005})
    key = {"WR":"separation_at_pass_forward","TE":"separation_at_pass_forward","DL":"player_get_off","DB":"cushion"}[P]
    k = dp[dp[key].notna()].groupby("nfl_id").size()
    print(f"key outcome {key}: rows {int(k.sum())}, players with data {len(k)}; >=20: {(k>=20).sum()}, >=50: {(k>=50).sum()}, >=100: {(k>=100).sum()}, median {k.median():.0f}")
    if P in ("WR","TE"):
        r = dp[dp.route_ran.notna()].groupby("nfl_id").size()
        print(f"route_ran rows: players {len(r)}, >=50: {(r>=50).sum()}, >=100: {(r>=100).sum()}, >=200: {(r>=200).sum()}")
    if P=="DL":
        for c in ["sack","time_to_qb_hurry","time_to_pressure"]:
            s = dp.groupby("nfl_id")[c].apply(lambda x: x.notna().sum()); print(f"  {c}: non-null events total {int(s.sum())}, players with >=5: {(s>=5).sum()}")
        pr = dp.groupby("nfl_id").size(); print(f"  DL pass-play snaps per player: median {pr.median():.0f}, >=200: {(pr>=200).sum()}, >=400: {(pr>=400).sum()}")
    # drill repeatability
    c = ct[(ct.entity_type=="PLAYER")&(ct.nfl_id.map(pos)==P)].copy()
    c["drill"]=c.drill_name.replace({"PASS_PRO-MIRROR_DRILL":"PASS_PRO_MIRROR_DRILL","OVER_THE_SHOULDER_ADJUST":"OVER_SHOULDER_ADJUST"})
    c = c[c.drill_type.str.startswith("SKILL")]
    a = c.groupby(["drill","nfl_id"]).event_id.nunique().reset_index(name="n")
    s = a.groupby("drill").n.agg(players="size", ge2=lambda x:(x>=2).sum(), ge3=lambda x:(x>=3).sum())
    print(s[s.players>=60].to_string())

# OL PASS_RUSH_DROP: do the two attempts differ in direction?
c = ct[(ct.entity_type=="PLAYER")&(ct.drill_name=="PASS_RUSH_DROP")&(ct.nfl_id.map(pos)=="OL")].sort_values(["nfl_id","event_id","time"])
disp = c.groupby(["nfl_id","event_id"]).apply(lambda e: pd.Series({"dx":e.x.iloc[-1]-e.x.iloc[0],"dy":e.y.iloc[-1]-e.y.iloc[0]})).reset_index()
rows=[]
for nid, e in disp.groupby("nfl_id"):
    if len(e)>=2:
        a,b = e.iloc[0], e.iloc[1]
        n1=np.hypot(a.dx,a.dy); n2=np.hypot(b.dx,b.dy)
        rows.append(((a.dx*b.dx+a.dy*b.dy)/(n1*n2+1e-9), n1, n2))
cs = pd.DataFrame(rows, columns=["cos","len1","len2"])
print("\nOL PASS_RUSH_DROP attempt1 vs attempt2 displacement cosine: n=",len(cs)); print(cs.cos.describe().round(2).to_dict())
print("share cos>0.8:", round((cs.cos>0.8).mean(),2), "| cos<0:", round((cs.cos<0).mean(),2), "| median path length (yd):", round(cs.len1.median(),1))

The audit above covers DL, DB, WR and TE; the next cell covers OL pass-protection coverage and the spread of pressure rates.

In [ ]:
import sys
sys.argv = ["x", DATA]
# --- 02_ol_feasibility.py ---
import sys, pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
D = sys.argv[1]
pd.set_option("display.width", 220, "display.max_columns", 60, "display.max_rows", 100)
cr = pd.read_csv(f"{D}/combine_results.csv", na_values=["NA"])
ct = pd.read_csv(f"{D}/combine_tracking.csv", usecols=["draft_year","event_id","nfl_id","entity_type","drill_type","drill_name","attempt","time"])
pp = pd.read_csv(f"{D}/player_play.csv", low_memory=False)
games = pd.read_csv(f"{D}/games.csv")
ol = set(cr[cr.combine_position=="OL"].nfl_id)

# 1. drill repeatability for OL
c = ct[(ct.nfl_id.isin(ol)) & (ct.entity_type=="PLAYER") & (ct.drill_type=="SKILL_DRILLS_OL")]
c["drill"] = c.drill_name.replace({"PASS_PRO-MIRROR_DRILL":"PASS_PRO_MIRROR_DRILL"})
a = c.groupby(["drill","nfl_id"]).event_id.nunique().rename("n_events").reset_index()
print("OL drill coverage: players with >=1 and >=2 recorded attempts (events)")
print(a.groupby("drill").n_events.agg(players="size", ge2=lambda s:(s>=2).sum(), median="median"))
print("\nOL players with BOTH pass_rush_drop and mirror:", len(set(a[a.drill=="PASS_RUSH_DROP"].nfl_id) & set(a[a.drill=="PASS_PRO_MIRROR_DRILL"].nfl_id)))

# 2. game exposure and eligible-role missingness
ppo = pp[pp.nfl_id.isin(ol)].merge(games[["game_id","season","season_type"]], on="game_id", how="left")
reg = ppo[ppo.season_type=="REG"]
print("\nOL player-play rows (REG):", len(reg), "| players:", reg.nfl_id.nunique())
print("lined_up_position:", reg.lined_up_position.value_counts().to_dict())
passes = reg[reg.pass_result.notna() & (reg.pass_result!="")]
print("\nOL rows on pass plays:", len(passes))
cols = ["pressure_allowed","sack_allowed","time_to_pressure_allowed","dropback_duration","pass_rushers_encountered","peak_pressure_probability_allowed","unblocked_pressure","quick_pressure"]
print("non-null % WITHIN OL pass-play rows:"); print((passes[cols].notna().mean()*100).round(1).to_dict())
pa = passes[passes.pressure_allowed.notna()]
pa["pressure_allowed"] = pa.pressure_allowed.astype(str).str.upper().isin(["TRUE","1","1.0"])
print("\npressure_allowed value counts (raw):", passes.pressure_allowed.value_counts(dropna=False).to_dict())
per = pa.groupby("nfl_id").agg(snaps=("pressure_allowed","size"), press=("pressure_allowed","mean"))
print("\npass-block snaps per OL player:", per.snaps.describe().round(0).to_dict())
for k in (50,100,200,400): print(f" players with >= {k} pass-block snaps: {(per.snaps>=k).sum()} of {len(ol)}")
print("overall pressure-allowed rate:", round(pa.pressure_allowed.mean(),3))
# between-player signal: expected binomial sd vs observed
p=pa.pressure_allowed.mean(); big=per[per.snaps>=100]
print(f"observed sd of player rate (>=100 snaps): {big.press.std():.3f} | pure-noise binomial sd: {np.sqrt((p*(1-p)/big.snaps).mean()):.3f}")
print("\nseason split of pass-block snaps:"); print(pa.merge(games[['game_id','season']],on='game_id').season_x.value_counts().to_dict() if 'season_x' in pa.merge(games[['game_id','season']],on='game_id').columns else pa.season.value_counts().to_dict())
print("\nOL pass-block snaps by draft class:")
dy = cr.set_index("nfl_id").draft_year
print(per.join(dy).groupby("draft_year").snaps.agg(["size","median","sum"]))

## 2. Sanity check: is the Combine sensor pipeline measuring speed correctly?
Tracked top speed in the 40-yard dash should track the official time. (Tracked 40 times are not comparable to official times because the clips start with the player already in motion, so we use speed levels, not times.)

In [ ]:
import sys
sys.argv = ["x", DATA]
# --- 07_sanity_speed.py ---
import sys, pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
D = sys.argv[1]
cr = pd.read_csv(f"{D}/combine_results.csv", na_values=["NA"]).set_index("nfl_id")
ct = pd.read_csv(f"{D}/combine_tracking.csv")
p = ct[ct.entity_type=="PLAYER"]
# 1) forty-yard dash: tracked max speed vs official forty
fd = p[p.drill_name=="FORTY_YARD_DASH"].groupby(["nfl_id","event_id"]).agg(vmax=("s","max"), n=("s","size"), dur=("time","size")).reset_index()
fd = fd.groupby("nfl_id").vmax.max().to_frame().join(cr[["forty","ten_yd_split","combine_position"]]).dropna()
print("FORTY tracked max speed vs official forty, corr by position:")
print(fd.groupby("combine_position").apply(lambda g: pd.Series({"n":len(g),"r_vmax_forty":g.vmax.corr(g.forty),"mean_vmax":g.vmax.mean()})).round(2))
# 2) WR route drills: top speed per player-drill
wr = set(cr[cr.combine_position=="WR"].index)
w = p[p.nfl_id.isin(wr)&(p.drill_type=="SKILL_DRILLS_WR")&(~p.drill_name.isin(["GAUNTLET_DRILL"]))]
t = w.groupby(["nfl_id","drill_name","event_id"]).s.max().reset_index().sort_values("event_id").groupby(["nfl_id","drill_name"]).head(1)
t["z"] = t.groupby("drill_name").s.transform(lambda s:(s-s.mean())/s.std())
cnt=t.groupby("nfl_id").size(); el=cnt[cnt>=6].index
rng=np.random.default_rng(1); rs=[]
for _ in range(200):
    a=[];b=[]
    for nid,g in t[t.nfl_id.isin(el)].groupby("nfl_id"):
        v=g.z.values; i=rng.permutation(len(v)); h=len(v)//2; a.append(v[i[:h]].mean()); b.append(v[i[h:2*h]].mean())
    rs.append(np.corrcoef(a,b)[0,1])
r=np.mean(rs); print(f"\nWR route top-speed split-half r={r:.2f}, SB={2*r/(1+r):.2f} (n players={len(el)})")
pm = t.groupby("nfl_id").z.mean().to_frame("topspeed_z").join(cr[["forty","ten_yd_split"]]).dropna()
print("corr(top speed in routes, forty) =", round(pm.topspeed_z.corr(pm.forty),2), "| n", len(pm))
# 3) timestamps/sampling sanity: dt distribution in a route event
g = w[w.event_id==w.event_id.iloc[0]]
print("\nrows in first route event:", len(g), "| dt:", pd.to_datetime(g.time, format="mixed").diff().dt.total_seconds().round(2).value_counts().head(3).to_dict())
print("s vs dis*10 check (corr):", round(np.corrcoef(g.s, g.dis*10)[0,1],3), "| mean s", round(g.s.mean(),2), "mean dis*10", round((g.dis*10).mean(),2))

## 3. Combine tracking features from the 40-yard dash
Downfield position (not the `dis` column, which undercounts distance) gives speed levels (10/20/30 yd, top speed) and shape features (peak acceleration, yards to 95% of top speed, time of top speed, speed maintenance).

In [ ]:
import sys
sys.argv = ["x", DATA, OUTDIR + "/forty_run_features.csv"]
# --- 08_forty_profile.py ---
import sys, pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
D, OUT = sys.argv[1], sys.argv[2]
pd.set_option("display.width", 200, "display.max_columns", 40)
cr = pd.read_csv(f"{D}/combine_results.csv", na_values=["NA"]).set_index("nfl_id")
ct = pd.read_csv(f"{D}/combine_tracking.csv")
p = ct[(ct.entity_type=="PLAYER")&(ct.drill_name=="FORTY_YARD_DASH")].copy()
p["t"] = pd.to_datetime(p.time, format="mixed")
def sm(v,k=3): return pd.Series(v).rolling(k,center=True,min_periods=1).mean().values
rows=[]
for (eid,nid),g in p.sort_values("t").groupby(["event_id","nfl_id"]):
    s=sm(g.s.values); n=len(s)
    if n<30: continue
    d=g.x.values-g.x.values[0]               # downfield yards (dis column undercounts)
    on = np.argmax(s>0.8) if (s>0.8).any() else None
    if on is None: continue
    d=d-d[on]; i40=np.argmax(d>=38) if (d>=38).any() else None
    if i40 is None or i40<=on: continue
    S=s[on:i40+1]; Dd=d[on:i40+1]-d[on]; T=np.arange(len(S))*0.1
    vmax=S.max(); acc=np.gradient(S)/0.1
    at=lambda yd: S[np.argmax(Dd>=yd)] if (Dd>=yd).any() else np.nan
    tt=lambda yd: T[np.argmax(Dd>=yd)] if (Dd>=yd).any() else np.nan
    rows.append(dict(nfl_id=nid,event_id=eid,t_event=g.t.iloc[0],t10=tt(10),t20=tt(20),t40=T[-1],v10=at(10),v20=at(20),v30=at(30),v40=S[-1],vmax=vmax,
        peak_acc=acc[:20].max(), d95=Dd[np.argmax(S>=0.95*vmax)], t_vmax=T[np.argmax(S)], maint=S[-1]/vmax))
f=pd.DataFrame(rows).sort_values("t_event")
f["run"]=f.groupby("nfl_id").cumcount()+1
f.to_csv(OUT,index=False)
print("runs:",len(f),"| players:",f.nfl_id.nunique(),"| runs per player:",f.groupby("nfl_id").size().value_counts().to_dict())
print("tracked t40 vs official forty corr:", round(f.groupby("nfl_id").t40.min().to_frame().join(cr.forty).dropna().corr().iloc[0,1],2))
feats=["t10","t40","v10","v20","v30","vmax","peak_acc","d95","t_vmax","maint"]
two=f[f.run<=2].pivot(index="nfl_id",columns="run",values=feats).dropna()
print("\ntest-retest (run1 vs run2) r, n=%d:"%len(two))
for m in feats: print(f"  {m:9s} r={np.corrcoef(two[(m,1)],two[(m,2)])[0,1]:.2f}")
P=f.groupby("nfl_id")[feats].mean().join(cr[["forty","ten_yd_split","combine_weight","combine_position"]]).dropna(subset=["forty","ten_yd_split","combine_weight"])
X=np.c_[np.ones(len(P)),P.forty,P.ten_yd_split,P.combine_weight]
print("\nR^2 of tracked feature on official forty+ten_yd+weight (n=%d), i.e. share already explained by conventional numbers:"%len(P))
for m in feats:
    y=P[m].values; b=np.linalg.lstsq(X,y,rcond=None)[0]; print(f"  {m:9s} R2={1-((y-X@b)**2).sum()/((y-y.mean())**2).sum():.2f}")

## 4. Negative result worth stating: route-break features at the Combine have no cross-route reliability
Each receiver runs each route once, so a 'route-break' feature would have to repeat across *different* routes. It does not, so it cannot be a stable player trait.

In [ ]:
import sys
sys.argv = ["x", DATA, OUTDIR + "/wr_rep_features.csv"]
# --- 05_wr_rep_features.py ---
import sys, pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
D, OUT = sys.argv[1], sys.argv[2]
pd.set_option("display.width", 220, "display.max_columns", 60, "display.max_rows", 120)
cr = pd.read_csv(f"{D}/combine_results.csv", na_values=["NA"])
wr = set(cr[cr.combine_position=="WR"].nfl_id)
ct = pd.read_csv(f"{D}/combine_tracking.csv")
ct = ct[(ct.entity_type=="PLAYER") & ct.nfl_id.isin(wr) & (ct.drill_type=="SKILL_DRILLS_WR")].copy()
ct["drill"] = ct.drill_name.replace({"OVER_THE_SHOULDER_ADJUST":"OVER_SHOULDER_ADJUST"})
ct = ct[~ct.drill.isin(["GAUNTLET_DRILL","OVER_SHOULDER_ADJUST"])]
ct["t"] = pd.to_datetime(ct.time, format="mixed")
def smooth(v, k=3): return pd.Series(v).rolling(k, center=True, min_periods=1).mean().values
rows=[]
for (eid,nid), g in ct.sort_values("t").groupby(["event_id","nfl_id"]):
    s = g.s.values; n=len(s)
    if n < 20: continue
    on = np.argmax(s>1.0) if (s>1.0).any() else None
    if on is None or on > n-15: continue
    th = np.degrees(np.unwrap(np.radians(smooth(g.dir.values))))
    om = np.gradient(th)/0.1                      # deg/s
    om_s = smooth(om)
    acc = smooth(np.gradient(smooth(s))/0.1)      # yd/s^2 from speed
    lo, hi = on+4, n-4
    if hi<=lo: continue
    c = lo + int(np.argmax(np.abs(om_s[lo:hi])))
    a0, a1 = max(on, c-8), min(n-1, c+8)
    turn = abs(th[a1]-th[a0])
    pre = s[max(on,c-10):max(on+1,c-4)].mean()
    smin = s[max(on,c-6):min(n,c+7)].min()
    dec = -acc[max(on,c-10):c+1].min()
    rea = acc[c:min(n,c+11)].max()
    exitv = s[min(n-1,c+10)]
    half = np.abs(om_s) >= 0.5*np.abs(om_s[c])
    dur = half[max(0,c-8):min(n,c+9)].sum()*0.1
    rows.append(dict(drill=g.drill.iloc[0], nfl_id=nid, event_id=eid, attempt=g.attempt.iloc[0], draft_year=g.draft_year.iloc[0],
        n=n, onset=on, c_idx=c, t_break=(c-on)*0.1, turn_deg=turn, peak_turn_rate=abs(om_s[c]), stem_speed=pre, min_speed=smin,
        retention=smin/pre if pre>0 else np.nan, decel_pre=dec, reaccel_post=rea, exit_speed=exitv, turn_dur=dur, top_speed=s.max(),
        x_break=g.x.values[c], y_break=g.y.values[c]))
f = pd.DataFrame(rows); f.to_csv(OUT, index=False)
print("reps:", len(f), "| players:", f.nfl_id.nunique())
sm = f.groupby("drill").agg(reps=("nfl_id","size"), turn=("turn_deg","median"), t_break=("t_break","median"), stem=("stem_speed","median"),
    retention=("retention","median"), decel=("decel_pre","median"), reaccel=("reaccel_post","median"), exitv=("exit_speed","median"), top=("top_speed","median"))
print(sm.round(2))
print("\nshare of reps with turn>=40 deg by drill:"); print((f.assign(b=f.turn_deg>=40).groupby("drill").b.mean()).round(2).to_dict())

In [ ]:
import sys
sys.argv = ["x", DATA, OUTDIR + "/wr_rep_features.csv"]
# --- 06_wr_reliability.py ---
import sys, os, pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
D, F = sys.argv[1], sys.argv[2]
pd.set_option("display.width", 200, "display.max_columns", 40)
cr = pd.read_csv(f"{D}/combine_results.csv", na_values=["NA"]).set_index("nfl_id")
f = pd.read_csv(F)
f["drill"]=f.drill.replace({"SPEED_OUT_LEFT":"SPEED_OUT_ROUTE_LEFT"})
keep = ["COMEBACK_ROUTE_RIGHT","DAGGER_ROUTE_LEFT","POST_CORNER_ROUTE_LEFT","SLANT_ROUTE_LEFT","SLOT_SAIL_ROUTE_RIGHT","SLOT_STRIKE_ROUTE_LEFT","SPEED_OUT_ROUTE_LEFT","CURL_ROUTE_RIGHT"]
f = f[f.drill.isin(keep) & (f.turn_deg>=40)].copy()
f = f.sort_values("attempt").groupby(["nfl_id","drill"]).head(1)   # one rep per player-drill
f["speed_regain"] = f.exit_speed - f.min_speed
metrics = ["exit_speed","min_speed","peak_turn_rate","turn_dur","speed_regain","reaccel_post"]
z = f.copy()
for m in metrics: z[m] = z.groupby("drill")[m].transform(lambda s:(s-s.mean())/s.std())
cnt = z.groupby("nfl_id").size()
elig = cnt[cnt>=4].index
print("break reps:", len(z), "| players with >=4 break drills:", len(elig), "| median drills per player:", cnt.median())
rng = np.random.default_rng(7)
def sb(r): return 2*r/(1+r)
print("\nsplit-half reliability across routes (200 random splits of each player's drills), Spearman-Brown:")
res={}
for m in metrics:
    rs=[]
    for _ in range(200):
        a=[];b=[]
        for nid,g in z[z.nfl_id.isin(elig)].groupby("nfl_id"):
            v=g[m].dropna().values
            if len(v)<4: continue
            idx=rng.permutation(len(v)); h=len(v)//2
            a.append(v[idx[:h]].mean()); b.append(v[idx[h:2*h]].mean())
        rs.append(np.corrcoef(a,b)[0,1])
    res[m]=(np.mean(rs), sb(np.mean(rs)))
    print(f"  {m:15s} r={res[m][0]:.2f}  SB={res[m][1]:.2f}")
P = z.groupby("nfl_id")[metrics].mean().loc[elig].join(cr[["forty","ten_yd_split","combine_weight","vertical","broad_jump","draft_year"]])
print("\ncorrelation of player-level mean z with conventional measures (n=%d):"%len(P))
print(P[metrics+["forty","ten_yd_split","combine_weight","vertical","broad_jump"]].corr().loc[metrics,["forty","ten_yd_split","combine_weight","vertical","broad_jump"]].round(2))
# how much of each metric is NOT explained by forty+weight (R^2)
import numpy.linalg as la
print("\nR^2 of metric on forty+ten_yd+weight (complete cases):")
Q=P.dropna(subset=["forty","ten_yd_split","combine_weight"])
X=np.c_[np.ones(len(Q)),Q.forty,Q.ten_yd_split,Q.combine_weight]
for m in metrics:
    y=Q[m].values; b=la.lstsq(X,y,rcond=None)[0]; r2=1-((y-X@b)**2).sum()/((y-y.mean())**2).sum(); print(f"  {m:15s} R2={r2:.2f}  (n={len(Q)})")
z.to_csv(os.path.join(os.path.dirname(F),"wr_break_reps_z.csv"), index=False)
P.to_csv(os.path.join(os.path.dirname(F),"wr_player_break_features.csv"))

## 5. NFL outcomes: in-game top speed per play
Max tracked speed on each regular-season play for the drafted players (split-half reliability of the per-player figure is ~0.98).

In [ ]:
import sys
sys.argv = ["x", DATA, OUTDIR + "/ingame_speed.csv"]
# --- 09_ingame_speed.py ---
import sys, pandas as pd, numpy as np, warnings, time
warnings.filterwarnings("ignore")
D, OUT = sys.argv[1], sys.argv[2]
t0=time.time()
g = pd.read_csv(f"{D}/games.csv"); reg=set(g[g.season_type=="REG"].game_id)
parts=[]
for y in (2023,2024,2025):
    for ch in pd.read_csv(f"{D}/game_tracking_{y}.csv", usecols=["game_id","play_id","nfl_id","s","a"], chunksize=3_000_000):
        ch=ch[ch.game_id.isin(reg)]
        parts.append(ch.groupby(["game_id","play_id","nfl_id"]).agg(smax=("s","max"),amax=("a","max"),n=("s","size")).reset_index())
    print(y,"done",round(time.time()-t0),"s",flush=True)
p=pd.concat(parts).groupby(["game_id","play_id","nfl_id"]).agg(smax=("smax","max"),amax=("amax","max"),n=("n","sum")).reset_index()
p.to_csv(OUT.replace(".csv","_play.csv"),index=False)
p=p[p.n>=20]
print("player-plays:",len(p),"| players:",p.nfl_id.nunique(),flush=True)
def summ(d): return pd.Series({"n_plays":len(d),"s_p95":d.smax.quantile(.95),"s_p99":d.smax.quantile(.99),"a_p95":d.amax.quantile(.95)})
P=p.groupby("nfl_id").apply(summ)
for nm,mask in (("odd",p.game_id%2==1),("even",p.game_id%2==0)):
    Q=p[mask].groupby("nfl_id").apply(summ); P[f"s_p95_{nm}"]=Q.s_p95; P[f"a_p95_{nm}"]=Q.a_p95
P.to_csv(OUT)
print(P.describe().round(2).to_string())
ok=P[P.n_plays>=100].dropna(subset=["s_p95_odd","s_p95_even"])
print("split-half (odd/even games) corr, players>=100 plays (n=%d): s_p95 r=%.2f | a_p95 r=%.2f"%(len(ok),ok.s_p95_odd.corr(ok.s_p95_even),ok.a_p95_odd.corr(ok.a_p95_even)))
print("done",round(time.time()-t0),"s",flush=True)

## 6. Test 1 (pre-set): do tracking *shape* features add predictive value beyond the stopwatch?
Cross-validated R² gain over a baseline of official 40 time, 10-yd split and weight. Expectation recorded beforehand: small. Result: no detectable gain.

In [ ]:
import sys
sys.argv = ["x", DATA, OUTDIR]
# --- 10_translation_test.py ---
import sys, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
D, O = sys.argv[1], sys.argv[2]
rng = np.random.default_rng(2027)
cr = pd.read_csv(f"{D}/combine_results.csv", na_values=["NA"]).set_index("nfl_id")
f = pd.read_csv(f"{O}/forty_run_features.csv")
F = f.groupby("nfl_id")[["maint","peak_acc","d95","vmax"]].mean()
sp = pd.read_csv(f"{O}/ingame_speed.csv").set_index("nfl_id")
games = pd.read_csv(f"{D}/games.csv")
pp = pd.read_csv(f"{D}/player_play.csv", low_memory=False).merge(games[["game_id","season_type"]], on="game_id")
pp = pp[pp.season_type=="REG"].copy(); pp["pos"] = pp.nfl_id.map(cr.combine_position)
BASE=["forty","ten_yd_split","combine_weight"]; FEAT=["maint","peak_acc","d95"]
T = cr[BASE+["combine_position"]].join(F).dropna(subset=BASE+FEAT)
print("players with baseline + 40-dash shape features:", len(T), T.combine_position.value_counts().to_dict())

def wls(X,y,w):
    sw=np.sqrt(w); return np.linalg.lstsq(X*sw[:,None], y*sw, rcond=None)[0]
def wr2(y,yh,w,ybar): return 1-(w*(y-yh)**2).sum()/(w*(y-ybar)**2).sum()
def zs(M): return (M-M.mean(0))/M.std(0)
def design(df, pooled):
    B=zs(df[BASE].values); Fm=zs(df[FEAT].values)
    if pooled: B=np.c_[B, pd.get_dummies(df.combine_position,drop_first=True).values.astype(float)]
    one=np.ones((len(df),1)); return np.c_[one,B], Fm
def pcorr(Xb,Fm,y,w):
    out=[]
    ry=y-Xb@wls(Xb,y,w)
    for j in range(Fm.shape[1]):
        rf=Fm[:,j]-Xb@wls(Xb,Fm[:,j],w)
        a=np.average(rf,weights=w); b=np.average(ry,weights=w)
        out.append(((w*(rf-a)*(ry-b)).sum())/np.sqrt((w*(rf-a)**2).sum()*(w*(ry-b)**2).sum()))
    return np.array(out)
def analyze(name, df, y, w, pooled=False, B=1000):
    Xb,Fm=design(df,pooled); Xf=np.c_[Xb,Fm]; n=len(df); y=np.asarray(y,float); w=np.asarray(w,float)
    yb=np.average(y,weights=w)
    r2b=wr2(y,Xb@wls(Xb,y,w),w,yb); r2f=wr2(y,Xf@wls(Xf,y,w),w,yb); d_in=r2f-r2b
    cvb=[];cvf=[]
    for rep in range(20):
        idx=rng.permutation(n); folds=np.array_split(idx,5); sb=sf=sst=0
        for k in range(5):
            te=folds[k]; tr=np.setdiff1d(idx,te); ybar=np.average(y[tr],weights=w[tr])
            pb=Xb[te]@wls(Xb[tr],y[tr],w[tr]); pf=Xf[te]@wls(Xf[tr],y[tr],w[tr])
            sb+=(w[te]*(y[te]-pb)**2).sum(); sf+=(w[te]*(y[te]-pf)**2).sum(); sst+=(w[te]*(y[te]-ybar)**2).sum()
        cvb.append(1-sb/sst); cvf.append(1-sf/sst)
    cvb,cvf=np.mean(cvb),np.mean(cvf)
    cnt=0
    for _ in range(1000):
        Fp=Fm[rng.permutation(n)]; Xp=np.c_[Xb,Fp]
        dp=wr2(y,Xp@wls(Xp,y,w),w,yb)-r2b; cnt+= dp>=d_in
    pval=(1+cnt)/1001
    pc=pcorr(Xb,Fm,y,w); bs=[]
    for _ in range(B):
        i=rng.integers(0,n,n); bs.append(pcorr(Xb[i],Fm[i],y[i],w[i]))
    lo,hi=np.nanpercentile(bs,[2.5,97.5],axis=0)
    print(f"\n[{name}] n={n}  baseline R2(in-sample)={r2b:.3f}  +shape: in-sample gain={d_in:.3f}  CV R2 base={cvb:.3f} full={cvf:.3f} CV gain={cvf-cvb:+.3f}  permutation p={pval:.3f}")
    print("   partial corr (controlling baseline), 95% player-bootstrap CI: "+" | ".join(f"{FEAT[j]} {pc[j]:+.2f} [{lo[j]:+.2f},{hi[j]:+.2f}]" for j in range(3)))
    return dict(outcome=name,n=n,base_r2=r2b,gain_in=d_in,cv_base=cvb,cv_full=cvf,cv_gain=cvf-cvb,perm_p=pval,**{f"pc_{FEAT[j]}":pc[j] for j in range(3)})
res=[]
# ---- O1 in-game top speed (z within position)
S=sp[sp.n_plays>=100].join(T,how="inner")
S["y"]=S.groupby("combine_position").s_p95.transform(lambda s:(s-s.mean())/s.std())
print("\nSANITY: corr(in-game p95 top speed, combine tracked vmax) within position:", S.groupby("combine_position").apply(lambda g: round(g.s_p95.corr(g.vmax),2)).to_dict())
print("SANITY: corr(in-game top speed, official forty) within position:", S.groupby("combine_position").apply(lambda g: round(g.s_p95.corr(g.forty),2)).to_dict())
res.append(analyze("O1 in-game top speed (all positions, pooled)", S, S.y, np.ones(len(S)), pooled=True))
for P in ["WR","DB","DL","OL"]:
    Q=S[S.combine_position==P]
    if len(Q)>=40: res.append(analyze(f"O1 in-game top speed ({P})", Q, Q.y, np.ones(len(Q)), pooled=False, B=500))
# ---- O2 WR adjusted separation
w_=pp[(pp.pos=="WR")&pp.route_ran.notna()&pp.separation_at_pass_forward.notna()&pp.cushion.notna()].copy()
Xd=pd.get_dummies(w_[["route_ran","team_coverage_man_zone","lined_up_position"]].fillna("NA"),drop_first=True).astype(float); Xd["cushion"]=w_.cushion.values
A=np.c_[np.ones(len(Xd)),Xd.values]; bb=np.linalg.lstsq(A,w_.separation_at_pass_forward.values,rcond=None)[0]
w_["res"]=w_.separation_at_pass_forward.values-A@bb
def eb(df,col,minn,sig2=None):
    g=df.groupby("nfl_id")[col].agg(["mean","size"]); g=g[g["size"]>=minn]
    s2=df[col].var() if sig2 is None else sig2
    tau2=max(g["mean"].var()-(s2/g["size"]).mean(),1e-4); g["w"]=1/(tau2+s2/g["size"]); return g,tau2,s2
g2,tau2,s2=eb(w_,"res",50)
for nm,mk in (("odd",w_.game_id%2==1),("even",w_.game_id%2==0)): g2[nm]=w_[mk].groupby("nfl_id").res.mean()
print(f"\nO2 WR separation residual: players>=50 routes {len(g2)}, between-player SD {np.sqrt(tau2):.2f} yd, play-level SD {np.sqrt(s2):.2f} yd, odd/even-game split-half r={g2.odd.corr(g2.even):.2f}")
Q=g2.join(T,how="inner"); Q=Q[Q.combine_position=="WR"]
res.append(analyze("O2 WR adjusted separation", Q, Q["mean"], Q.w, pooled=False))
# ---- O3 DL get-off
d_=pp[(pp.pos=="DL")].copy(); d_["go"]=pd.to_numeric(d_.player_get_off,errors="coerce"); d_=d_[d_.go.notna()]
print("\nO3 DL player_get_off distribution:", d_.go.describe().round(3).to_dict())
g3,tau3,s3=eb(d_,"go",50)
for nm,mk in (("odd",d_.game_id%2==1),("even",d_.game_id%2==0)): g3[nm]=d_[mk].groupby("nfl_id").go.mean()
print(f"O3 players>=50 snaps {len(g3)}, between SD {np.sqrt(tau3):.3f}, play SD {np.sqrt(s3):.3f}, split-half r={g3.odd.corr(g3.even):.2f}")
Q=g3.join(T,how="inner"); Q=Q[Q.combine_position=="DL"]
res.append(analyze("O3 DL get-off", Q, Q["mean"], Q.w, pooled=False))
# ---- O4 OL pressure allowed (secondary)
o_=pp[(pp.pos=="OL")&pp.pressure_allowed.notna()].copy(); o_["pr"]=o_.pressure_allowed.astype(str).str.upper().isin(["TRUE","1","1.0"]).astype(float)
p0=o_.pr.mean(); g4,tau4,s4=eb(o_,"pr",100,sig2=p0*(1-p0))
for nm,mk in (("odd",o_.game_id%2==1),("even",o_.game_id%2==0)): g4[nm]=o_[mk].groupby("nfl_id").pr.mean()
print(f"\nO4 OL pressure rate {p0:.3f}: players>=100 snaps {len(g4)}, between SD {np.sqrt(tau4):.3f}, split-half r={g4.odd.corr(g4.even):.2f}")
Q=g4.join(T,how="inner"); Q=Q[Q.combine_position=="OL"]
res.append(analyze("O4 OL pressure allowed (secondary)", Q, Q["mean"], Q.w, pooled=False))
pd.DataFrame(res).to_csv(f"{O}/translation_results.csv",index=False)
print("\nsaved translation_results.csv")

## 7. Test 2 (pre-set rule): how much prior weight does a Combine-informed prior get?
Leave-one-player-out empirical-Bayes shrinkage. Pre-set claim rule (written down in a dated note before this analysis): the CI for MSE(position mean) − MSE(Combine prior) must exclude 0 at both k = 0 and k = 25 early snaps.

In [ ]:
import sys
sys.argv = ["x", DATA, OUTDIR]
# --- 12_snaps_equivalent.py ---
import sys, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
D, O = sys.argv[1], sys.argv[2]
rng = np.random.default_rng(2027)
BASE=["forty","ten_yd_split","combine_weight"]; FEAT=["maint","peak_acc","d95"]; KS=[0,10,25,50,100]; LATE=100
cr = pd.read_csv(f"{D}/combine_results.csv", na_values=["NA"]).set_index("nfl_id")
f = pd.read_csv(f"{O}/forty_run_features.csv"); F = f.groupby("nfl_id")[FEAT].mean()
T = cr[BASE+["combine_position"]].join(F)
games = pd.read_csv(f"{D}/games.csv")
pp = pd.read_csv(f"{D}/player_play.csv", low_memory=False).merge(games[["game_id","season_type"]], on="game_id")
pp = pp[pp.season_type=="REG"].copy(); pp["pos"]=pp.nfl_id.map(cr.combine_position)
tabs={}
d=pp[pp.pos=="DL"].copy(); d["y"]=pd.to_numeric(d.player_get_off,errors="coerce"); tabs["DL get-off (lower=faster)"]=("DL",d[d.y.notna()])
w=pp[(pp.pos=="WR")&pp.route_ran.notna()&pp.separation_at_pass_forward.notna()&pp.cushion.notna()].copy()
Xd=pd.get_dummies(w[["route_ran","team_coverage_man_zone","lined_up_position"]].fillna("NA"),drop_first=True).astype(float); Xd["cushion"]=w.cushion.values
A=np.c_[np.ones(len(Xd)),Xd.values]; w["y"]=w.separation_at_pass_forward.values-A@np.linalg.lstsq(A,w.separation_at_pass_forward.values,rcond=None)[0]
tabs["WR adjusted separation"]=("WR",w)
o=pp[(pp.pos=="OL")&pp.pressure_allowed.notna()].copy(); o["y"]=o.pressure_allowed.astype(str).str.upper().isin(["TRUE","1","1.0"]).astype(float); tabs["OL pressure allowed"]=("OL",o)
sp=pd.read_csv(f"{O}/ingame_speed_play.csv"); sp=sp[sp.n>=20].copy(); sp["pos"]=sp.nfl_id.map(cr.combine_position); sp["y"]=sp.smax
for P in ["WR","DL","DB","OL"]: tabs[f"{P} in-game top speed"]=(P,sp[sp.pos==P])
def zs(M,mu,sd): return (M-mu)/sd
def fit_prior(ids, stats, cols, sig2):
    s=stats.loc[ids]; m=s["mean"].values; n=s["n"].values
    if cols:
        X=T.loc[ids,cols].values.astype(float); mu,sd=X.mean(0),X.std(0)+1e-9; X=np.c_[np.ones(len(ids)),zs(X,mu,sd)]
    else: X=np.ones((len(ids),1)); mu=sd=None
    tt=max(np.var(m)-np.mean(sig2/n),1e-9); wts=1/(0.5*tt+sig2/n)
    sw=np.sqrt(wts); b=np.linalg.lstsq(X*sw[:,None],m*sw,rcond=None)[0]
    r=m-X@b; p=X.shape[1]; N=len(ids)
    tau2=max((wts*(r**2*N/max(N-p,1)-sig2/n)).sum()/wts.sum(),1e-9)
    return b,mu,sd,tau2
def predict(b,mu,sd,pid,cols):
    if cols: x=np.r_[1,zs(T.loc[pid,cols].values.astype(float),mu,sd)]
    else: x=np.array([1.0])
    return float(x@b)
out=[]
for name,(P,df) in tabs.items():
    df=df.sort_values(["nfl_id","game_id","play_id"])
    ok=T[(T.combine_position==P)].dropna(subset=BASE+FEAT).index
    df=df[df.nfl_id.isin(ok)]
    g=df.groupby("nfl_id").y
    stats=pd.DataFrame({"n":g.size(),"mean":g.mean(),"var":g.var(ddof=1)})
    pool=stats[stats.n>=30].index.tolist(); test=stats[stats.n>=KS[-1]+LATE].index.tolist()
    if len(test)<15: print(f"[{name}] too few test players ({len(test)})"); continue
    ys={i:df[df.nfl_id==i].y.values for i in test}
    sig2_all=((stats.loc[pool,"n"]-1)*stats.loc[pool,"var"]).sum()/(stats.loc[pool,"n"]-1).sum()
    # full-pool parameters
    _,_,_,t_pos=fit_prior(pool,stats,[],sig2_all); _,_,_,t_base=fit_prior(pool,stats,BASE,sig2_all); _,_,_,t_shape=fit_prior(pool,stats,BASE+FEAT,sig2_all)
    t_base=max(t_base,0.02*t_pos); t_shape=max(t_shape,0.02*t_pos)
    E={k:{m:[] for m in ("raw","pos","base","shape")} for k in KS}; truth=[]
    for pid in test:
        rest=[i for i in pool if i!=pid]
        rs=stats.loc[rest]; sig2=((rs.n-1)*rs["var"]).sum()/(rs.n-1).sum()
        pri={}
        for arm,cols in (("pos",[]),("base",BASE),("shape",BASE+FEAT)):
            b,mu,sd,t2=fit_prior(rest,stats,cols,sig2); t2=max(t2,0.02*fit_prior(rest,stats,[],sig2)[3]) if arm!="pos" else t2
            pri[arm]=(predict(b,mu,sd,pid,cols),t2)
        y=ys[pid]; late=y[-LATE:].mean(); truth.append(late)
        for k in KS:
            ek=y[:k]
            for arm in ("pos","base","shape"):
                mu0,t2=pri[arm]
                post=mu0 if k==0 else (k/sig2*ek.mean()+mu0/t2)/(k/sig2+1/t2)
                E[k][arm].append((post-late)**2)
            E[k]["raw"].append(np.nan if k==0 else (ek.mean()-late)**2)
    n_t=len(test)
    print(f"\n[{name}] pool {len(pool)}, test {n_t} | sigma {np.sqrt(sig2_all):.3f}, tau_tot {np.sqrt(t_pos):.3f}, tau_after_combine {np.sqrt(t_base):.3f} | Combine explains {100*(1-t_base/t_pos):.0f}% of true between-player variance | n0 position-mean {sig2_all/t_pos:.0f} snaps, n0 Combine prior {sig2_all/t_base:.0f} snaps, with shape {sig2_all/t_shape:.0f}")
    print("   k   MSEraw   MSEpos   MSEcomb  MSEshape | pos-comb gain [95% CI]   | comb-shape gain [95% CI]")
    for k in KS:
        a={m:np.array(E[k][m]) for m in E[k]}
        def ci(x,y):
            d=x-y; bs=[d[rng.integers(0,n_t,n_t)].mean() for _ in range(2000)]; return d.mean(),np.percentile(bs,2.5),np.percentile(bs,97.5)
        g1=ci(a["pos"],a["base"]); g2=ci(a["base"],a["shape"])
        raw=np.nanmean(a["raw"]) if k>0 else np.nan
        print(f"  {k:3d}  {raw:7.4f}  {a['pos'].mean():7.4f}  {a['base'].mean():7.4f}  {a['shape'].mean():7.4f} | {g1[0]:+.4f} [{g1[1]:+.4f},{g1[2]:+.4f}] | {g2[0]:+.4f} [{g2[1]:+.4f},{g2[2]:+.4f}]")
        out.append(dict(outcome=name,k=k,n_test=n_t,pool=len(pool),sigma=np.sqrt(sig2_all),tau_tot=np.sqrt(t_pos),tau_comb=np.sqrt(t_base),share_explained=1-t_base/t_pos,n0_pos=sig2_all/t_pos,n0_comb=sig2_all/t_base,mse_raw=raw,mse_pos=a['pos'].mean(),mse_comb=a['base'].mean(),mse_shape=a['shape'].mean(),gain_pos_comb=g1[0],lo=g1[1],hi=g1[2],gain_comb_shape=g2[0],lo2=g2[1],hi2=g2[2]))
pd.DataFrame(out).to_csv(f"{O}/snaps_equivalent_results.csv",index=False); print("\nsaved")

## 8. Exploratory (post hoc): tracked-sensor prior vs official-stopwatch prior
Same players, same outcomes. Also reports CIs for each prior against the position average.

In [ ]:
import sys
sys.argv = ["x", DATA, OUTDIR]
# --- 13_sensor_vs_stopwatch.py ---
import sys, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
D, O = sys.argv[1], sys.argv[2]
rng=np.random.default_rng(2027)
STOP=["forty","ten_yd_split","combine_weight"]; SENS=["vmax","v10","combine_weight"]; KS=[0,10,25,50,100]; LATE=100
cr=pd.read_csv(f"{D}/combine_results.csv",na_values=["NA"]).set_index("nfl_id")
f=pd.read_csv(f"{O}/forty_run_features.csv"); F=f.groupby("nfl_id")[["vmax","v10"]].mean()
T=cr[["forty","ten_yd_split","combine_weight","combine_position"]].join(F)
print("COVERAGE by position (n players): stopwatch-complete | sensor-complete | either | sensor only (no complete stopwatch) | stopwatch only")
for P,g in T.groupby("combine_position"):
    s=g[STOP].notna().all(axis=1); e=g[SENS].notna().all(axis=1)
    print(f"  {P}: total {len(g)} | {s.sum()} | {e.sum()} | {(s|e).sum()} | {(e&~s).sum()} | {(s&~e).sum()}")
s=T[STOP].notna().all(axis=1); e=T[SENS].notna().all(axis=1)
print(f"  ALL: total {len(T)} | {s.sum()} | {e.sum()} | {(s|e).sum()} | sensor-only {(e&~s).sum()} | stopwatch-only {(s&~e).sum()}")
print("corr tracked vmax vs official forty (all):", round(T.vmax.corr(T.forty),2), "| tracked v10 vs official ten_yd:", round(T.v10.corr(T.ten_yd_split),2))
games=pd.read_csv(f"{D}/games.csv")
pp=pd.read_csv(f"{D}/player_play.csv",low_memory=False).merge(games[["game_id","season_type"]],on="game_id")
pp=pp[pp.season_type=="REG"].copy(); pp["pos"]=pp.nfl_id.map(cr.combine_position)
tabs={}
d=pp[pp.pos=="DL"].copy(); d["y"]=pd.to_numeric(d.player_get_off,errors="coerce"); tabs["DL get-off"]=("DL",d[d.y.notna()])
w=pp[(pp.pos=="WR")&pp.route_ran.notna()&pp.separation_at_pass_forward.notna()&pp.cushion.notna()].copy()
Xd=pd.get_dummies(w[["route_ran","team_coverage_man_zone","lined_up_position"]].fillna("NA"),drop_first=True).astype(float); Xd["cushion"]=w.cushion.values
A=np.c_[np.ones(len(Xd)),Xd.values]; w["y"]=w.separation_at_pass_forward.values-A@np.linalg.lstsq(A,w.separation_at_pass_forward.values,rcond=None)[0]; tabs["WR adj separation"]=("WR",w)
sp=pd.read_csv(f"{O}/ingame_speed_play.csv"); sp=sp[sp.n>=20].copy(); sp["pos"]=sp.nfl_id.map(cr.combine_position); sp["y"]=sp.smax
for P in ["WR","DL","DB","OL"]: tabs[f"{P} top speed"]=(P,sp[sp.pos==P])
def fit_prior(ids,stats,cols,sig2):
    s=stats.loc[ids]; m=s["mean"].values; n=s["n"].values
    if cols: X=T.loc[ids,cols].values.astype(float); mu,sd=X.mean(0),X.std(0)+1e-9; X=np.c_[np.ones(len(ids)),(X-mu)/sd]
    else: X=np.ones((len(ids),1)); mu=sd=None
    tt=max(np.var(m)-np.mean(sig2/n),1e-9); wts=1/(0.5*tt+sig2/n); sw=np.sqrt(wts); b=np.linalg.lstsq(X*sw[:,None],m*sw,rcond=None)[0]
    r=m-X@b; N=len(ids); p=X.shape[1]; tau2=max((wts*(r**2*N/max(N-p,1)-sig2/n)).sum()/wts.sum(),1e-9); return b,mu,sd,tau2
def pred(b,mu,sd,pid,cols):
    x=np.r_[1,(T.loc[pid,cols].values.astype(float)-mu)/sd] if cols else np.array([1.0]); return float(x@b)
res=[]
print("\nEXPLORATORY: stopwatch prior vs sensor prior, same players (LOO, late=last 100 plays). gain = MSE(stop)-MSE(sensor); positive favours sensor")
for name,(P,df) in tabs.items():
    df=df.sort_values(["nfl_id","game_id","play_id"])
    ok=T[(T.combine_position==P)].dropna(subset=STOP+SENS).index; df=df[df.nfl_id.isin(ok)]
    g=df.groupby("nfl_id").y; st=pd.DataFrame({"n":g.size(),"mean":g.mean(),"var":g.var(ddof=1)})
    pool=st[st.n>=30].index.tolist(); test=st[st.n>=KS[-1]+LATE].index.tolist()
    if len(test)<15: continue
    ys={i:df[df.nfl_id==i].y.values for i in test}
    sig2_all=((st.loc[pool,"n"]-1)*st.loc[pool,"var"]).sum()/(st.loc[pool,"n"]-1).sum()
    tp=fit_prior(pool,st,[],sig2_all)[3]; ts=max(fit_prior(pool,st,STOP,sig2_all)[3],0.02*tp); te=max(fit_prior(pool,st,SENS,sig2_all)[3],0.02*tp)
    E={k:{a:[] for a in ("pos","stop","sens")} for k in KS}
    for pid in test:
        rest=[i for i in pool if i!=pid]; rs=st.loc[rest]; sig2=((rs.n-1)*rs["var"]).sum()/(rs.n-1).sum(); tpos=fit_prior(rest,st,[],sig2)[3]
        pri={}
        for arm,cols in (("pos",[]),("stop",STOP),("sens",SENS)):
            b,mu,sd,t2=fit_prior(rest,st,cols,sig2); t2=max(t2,0.02*tpos) if arm!="pos" else t2; pri[arm]=(pred(b,mu,sd,pid,cols),t2)
        y=ys[pid]; late=y[-LATE:].mean()
        for k in KS:
            for arm in pri:
                mu0,t2=pri[arm]; post=mu0 if k==0 else (k/sig2*y[:k].mean()+mu0/t2)/(k/sig2+1/t2); E[k][arm].append((post-late)**2)
    n=len(test)
    print(f"\n[{name}] pool {len(pool)}, test {n} | Combine-explained true variance: stopwatch {100*(1-ts/tp):.0f}%, sensor {100*(1-te/tp):.0f}% | n0: stopwatch {sig2_all/ts:.0f}, sensor {sig2_all/te:.0f}")
    for k in (0,25,100):
        a={m:np.array(E[k][m]) for m in E[k]}; d=a["stop"]-a["sens"]; bs=[d[rng.integers(0,n,n)].mean() for _ in range(2000)]
        print(f"   k={k:3d}  MSE pos {a['pos'].mean():.4f} | stopwatch {a['stop'].mean():.4f} | sensor {a['sens'].mean():.4f} | gain {d.mean():+.4f} [{np.percentile(bs,2.5):+.4f},{np.percentile(bs,97.5):+.4f}]")
    for k in KS:
        a={m:np.array(E[k][m]) for m in E[k]}; d=a["stop"]-a["sens"]; bs=[d[rng.integers(0,n,n)].mean() for _ in range(1000)]
        d2=a["pos"]-a["sens"]; d3=a["pos"]-a["stop"]; ix=[rng.integers(0,n,n) for _ in range(2000)]
        b2=[d2[j].mean() for j in ix]; b3=[d3[j].mean() for j in ix]
        res.append(dict(outcome=name,k=k,n_test=n,mse_pos=a["pos"].mean(),mse_stop=a["stop"].mean(),mse_sens=a["sens"].mean(),gain=d.mean(),lo=np.percentile(bs,2.5),hi=np.percentile(bs,97.5),
            gain_sens_vs_pos=d2.mean(),lo_sens_vs_pos=np.percentile(b2,2.5),hi_sens_vs_pos=np.percentile(b2,97.5),gain_stop_vs_pos=d3.mean(),lo_stop_vs_pos=np.percentile(b3,2.5),hi_stop_vs_pos=np.percentile(b3,97.5),share_stop=1-ts/tp,share_sens=1-te/tp,n0_stop=sig2_all/ts,n0_sens=sig2_all/te))
pd.DataFrame(res).to_csv(f"{O}/sensor_vs_stopwatch.csv",index=False); print("\nsaved")

## 9. Robustness: multiplicity, seeds, and which Combine variable carries the prior
Holm correction across the seven outcomes (added after the fact), single-player influence on the gains, the claim rule under five other bootstrap seeds, an ablation of the prior (full / body weight only / 40 time + 10-yd split only), player-bootstrap ranges for the prior weights, and tracked-vs-official agreement within position.

In [ ]:
# Source of step 7 (12_snaps_equivalent.py), reused by the robustness checks below
SRC12 = 'import sys, numpy as np, pandas as pd, warnings\nwarnings.filterwarnings("ignore")\nD, O = sys.argv[1], sys.argv[2]\nrng = np.random.default_rng(2027)\nBASE=["forty","ten_yd_split","combine_weight"]; FEAT=["maint","peak_acc","d95"]; KS=[0,10,25,50,100]; LATE=100\ncr = pd.read_csv(f"{D}/combine_results.csv", na_values=["NA"]).set_index("nfl_id")\nf = pd.read_csv(f"{O}/forty_run_features.csv"); F = f.groupby("nfl_id")[FEAT].mean()\nT = cr[BASE+["combine_position"]].join(F)\ngames = pd.read_csv(f"{D}/games.csv")\npp = pd.read_csv(f"{D}/player_play.csv", low_memory=False).merge(games[["game_id","season_type"]], on="game_id")\npp = pp[pp.season_type=="REG"].copy(); pp["pos"]=pp.nfl_id.map(cr.combine_position)\ntabs={}\nd=pp[pp.pos=="DL"].copy(); d["y"]=pd.to_numeric(d.player_get_off,errors="coerce"); tabs["DL get-off (lower=faster)"]=("DL",d[d.y.notna()])\nw=pp[(pp.pos=="WR")&pp.route_ran.notna()&pp.separation_at_pass_forward.notna()&pp.cushion.notna()].copy()\nXd=pd.get_dummies(w[["route_ran","team_coverage_man_zone","lined_up_position"]].fillna("NA"),drop_first=True).astype(float); Xd["cushion"]=w.cushion.values\nA=np.c_[np.ones(len(Xd)),Xd.values]; w["y"]=w.separation_at_pass_forward.values-A@np.linalg.lstsq(A,w.separation_at_pass_forward.values,rcond=None)[0]\ntabs["WR adjusted separation"]=("WR",w)\no=pp[(pp.pos=="OL")&pp.pressure_allowed.notna()].copy(); o["y"]=o.pressure_allowed.astype(str).str.upper().isin(["TRUE","1","1.0"]).astype(float); tabs["OL pressure allowed"]=("OL",o)\nsp=pd.read_csv(f"{O}/ingame_speed_play.csv"); sp=sp[sp.n>=20].copy(); sp["pos"]=sp.nfl_id.map(cr.combine_position); sp["y"]=sp.smax\nfor P in ["WR","DL","DB","OL"]: tabs[f"{P} in-game top speed"]=(P,sp[sp.pos==P])\ndef zs(M,mu,sd): return (M-mu)/sd\ndef fit_prior(ids, stats, cols, sig2):\n    s=stats.loc[ids]; m=s["mean"].values; n=s["n"].values\n    if cols:\n        X=T.loc[ids,cols].values.astype(float); mu,sd=X.mean(0),X.std(0)+1e-9; X=np.c_[np.ones(len(ids)),zs(X,mu,sd)]\n    else: X=np.ones((len(ids),1)); mu=sd=None\n    tt=max(np.var(m)-np.mean(sig2/n),1e-9); wts=1/(0.5*tt+sig2/n)\n    sw=np.sqrt(wts); b=np.linalg.lstsq(X*sw[:,None],m*sw,rcond=None)[0]\n    r=m-X@b; p=X.shape[1]; N=len(ids)\n    tau2=max((wts*(r**2*N/max(N-p,1)-sig2/n)).sum()/wts.sum(),1e-9)\n    return b,mu,sd,tau2\ndef predict(b,mu,sd,pid,cols):\n    if cols: x=np.r_[1,zs(T.loc[pid,cols].values.astype(float),mu,sd)]\n    else: x=np.array([1.0])\n    return float(x@b)\nout=[]\nfor name,(P,df) in tabs.items():\n    df=df.sort_values(["nfl_id","game_id","play_id"])\n    ok=T[(T.combine_position==P)].dropna(subset=BASE+FEAT).index\n    df=df[df.nfl_id.isin(ok)]\n    g=df.groupby("nfl_id").y\n    stats=pd.DataFrame({"n":g.size(),"mean":g.mean(),"var":g.var(ddof=1)})\n    pool=stats[stats.n>=30].index.tolist(); test=stats[stats.n>=KS[-1]+LATE].index.tolist()\n    if len(test)<15: print(f"[{name}] too few test players ({len(test)})"); continue\n    ys={i:df[df.nfl_id==i].y.values for i in test}\n    sig2_all=((stats.loc[pool,"n"]-1)*stats.loc[pool,"var"]).sum()/(stats.loc[pool,"n"]-1).sum()\n    # full-pool parameters\n    _,_,_,t_pos=fit_prior(pool,stats,[],sig2_all); _,_,_,t_base=fit_prior(pool,stats,BASE,sig2_all); _,_,_,t_shape=fit_prior(pool,stats,BASE+FEAT,sig2_all)\n    t_base=max(t_base,0.02*t_pos); t_shape=max(t_shape,0.02*t_pos)\n    E={k:{m:[] for m in ("raw","pos","base","shape")} for k in KS}; truth=[]\n    for pid in test:\n        rest=[i for i in pool if i!=pid]\n        rs=stats.loc[rest]; sig2=((rs.n-1)*rs["var"]).sum()/(rs.n-1).sum()\n        pri={}\n        for arm,cols in (("pos",[]),("base",BASE),("shape",BASE+FEAT)):\n            b,mu,sd,t2=fit_prior(rest,stats,cols,sig2); t2=max(t2,0.02*fit_prior(rest,stats,[],sig2)[3]) if arm!="pos" else t2\n            pri[arm]=(predict(b,mu,sd,pid,cols),t2)\n        y=ys[pid]; late=y[-LATE:].mean(); truth.append(late)\n        for k in KS:\n            ek=y[:k]\n            for arm in ("pos","base","shape"):\n                mu0,t2=pri[arm]\n                post=mu0 if k==0 else (k/sig2*ek.mean()+mu0/t2)/(k/sig2+1/t2)\n                E[k][arm].append((post-late)**2)\n            E[k]["raw"].append(np.nan if k==0 else (ek.mean()-late)**2)\n    n_t=len(test)\n    print(f"\\n[{name}] pool {len(pool)}, test {n_t} | sigma {np.sqrt(sig2_all):.3f}, tau_tot {np.sqrt(t_pos):.3f}, tau_after_combine {np.sqrt(t_base):.3f} | Combine explains {100*(1-t_base/t_pos):.0f}% of true between-player variance | n0 position-mean {sig2_all/t_pos:.0f} snaps, n0 Combine prior {sig2_all/t_base:.0f} snaps, with shape {sig2_all/t_shape:.0f}")\n    print("   k   MSEraw   MSEpos   MSEcomb  MSEshape | pos-comb gain [95% CI]   | comb-shape gain [95% CI]")\n    for k in KS:\n        a={m:np.array(E[k][m]) for m in E[k]}\n        def ci(x,y):\n            d=x-y; bs=[d[rng.integers(0,n_t,n_t)].mean() for _ in range(2000)]; return d.mean(),np.percentile(bs,2.5),np.percentile(bs,97.5)\n        g1=ci(a["pos"],a["base"]); g2=ci(a["base"],a["shape"])\n        raw=np.nanmean(a["raw"]) if k>0 else np.nan\n        print(f"  {k:3d}  {raw:7.4f}  {a[\'pos\'].mean():7.4f}  {a[\'base\'].mean():7.4f}  {a[\'shape\'].mean():7.4f} | {g1[0]:+.4f} [{g1[1]:+.4f},{g1[2]:+.4f}] | {g2[0]:+.4f} [{g2[1]:+.4f},{g2[2]:+.4f}]")\n        out.append(dict(outcome=name,k=k,n_test=n_t,pool=len(pool),sigma=np.sqrt(sig2_all),tau_tot=np.sqrt(t_pos),tau_comb=np.sqrt(t_base),share_explained=1-t_base/t_pos,n0_pos=sig2_all/t_pos,n0_comb=sig2_all/t_base,mse_raw=raw,mse_pos=a[\'pos\'].mean(),mse_comb=a[\'base\'].mean(),mse_shape=a[\'shape\'].mean(),gain_pos_comb=g1[0],lo=g1[1],hi=g1[2],gain_comb_shape=g2[0],lo2=g2[1],hi2=g2[2]))\npd.DataFrame(out).to_csv(f"{O}/snaps_equivalent_results.csv",index=False); print("\\nsaved")\n'

In [ ]:
import sys
sys.argv = ["x", DATA, OUTDIR]
# --- 16_robustness.py ---
# Robustness for the pre-set claim rule: (a) different bootstrap seeds, (b) intersection-union p-values with Holm correction.
# Re-runs the exact logic of 12_snaps_equivalent.py, but keeps per-player squared-error differences (position-mean minus Combine prior).
import sys, io, contextlib, numpy as np, pandas as pd
D, O = sys.argv[1], sys.argv[2]
src = globals().get("SRC12") or open(__import__("os").path.join(__import__("os").path.dirname(__import__("os").path.abspath(__file__)), "12_snaps_equivalent.py")).read()   # SRC12 is set by the notebook
src = src.replace('pd.DataFrame(out).to_csv(f"{O}/snaps_equivalent_results.csv",index=False); print("\\nsaved")', 'pass')
src = src.replace("g1=ci(a[\"pos\"],a[\"base\"]);", "g1=ci(a[\"pos\"],a[\"base\"]); DIFFS[(name,k)]=a[\"pos\"]-a[\"base\"];")
DIFFS = {}
sys.argv = ["x", D, O]
with contextlib.redirect_stdout(io.StringIO()):
    exec(compile(src, "12_patched", "exec"), {"DIFFS": DIFFS, "__name__": "__main__"})
names = sorted({n for n, _ in DIFFS})
def boot(d, seed, B=20000):
    r = np.random.default_rng(seed); n = len(d); m = np.array([d[r.integers(0, n, n)].mean() for _ in range(B)])
    return m
rows = []
for nm in names:
    row = {"outcome": nm, "n_test": len(DIFFS[(nm, 0)])}
    met = []
    for seed in (1, 2, 3, 4, 5):
        ok = True
        for k in (0, 25):
            m = boot(DIFFS[(nm, k)], seed, 4000); ok &= np.percentile(m, 2.5) > 0
        met.append(ok)
    row["rule_met_in_seeds_of_5"] = sum(met)
    # one-sided bootstrap p for gain > 0 at k=0 and k=25; intersection-union p = max of the two
    p = {}
    for k in (0, 25):
        m = boot(DIFFS[(nm, k)], 11, 20000); p[k] = (np.sum(m <= 0) + 1) / (len(m) + 1)
    row.update(p_k0=p[0], p_k25=p[25], p_iu=max(p[0], p[25])); rows.append(row)
R = pd.DataFrame(rows).sort_values("p_iu").reset_index(drop=True); m = len(R)
R["holm_threshold"] = [0.05 / (m - i) for i in range(m)]
# Holm step-down
rej = []; stop = False
for i, r in R.iterrows():
    if not stop and r.p_iu <= r.holm_threshold: rej.append(True)
    else: stop = True; rej.append(False)
R["holm_reject_at_.05"] = rej
# Holm-adjusted p-values (step-down, monotone)
adj = []; run = 0.0
for i, r in R.iterrows():
    run = max(run, min(1.0, (m - i) * r.p_iu)); adj.append(run)
R["holm_adjusted_p"] = adj
pd.set_option("display.width", 220); print(R.round(4).to_string(index=False)); R.to_csv(f"{O}/robustness_holm.csv", index=False)
# Influence: does a single test player drive the gain? (drop-one means, and share of the total gain from the 3 largest contributors)
inf = []
for nm in names:
    for k in (0, 25):
        d = DIFFS[(nm, k)]; n = len(d); tot = d.sum(); loo = (tot - d) / (n - 1)
        top3 = np.sort(d)[::-1][:3].sum() / tot if tot > 0 else np.nan
        inf.append(dict(outcome=nm, k=k, n_test=n, mean_gain=d.mean(), min_dropone_gain=loo.min(), max_dropone_gain=loo.max(), top3_share_of_total=top3))
I = pd.DataFrame(inf); I.to_csv(f"{O}/influence.csv", index=False); print("\ninfluence of single test players on the gain (position mean minus Combine prior):"); print(I.round(4).to_string(index=False))

In [ ]:
import sys
sys.argv = ["x", DATA, OUTDIR]
# --- 17_ablation_and_uncertainty.py ---
# (a) Which Combine variable carries the prior? Ablation of the Combine prior (full / weight only / 40 time + 10-yd split only),
#     same players, same LOO code as 12_snaps_equivalent.py.  (b) Player-bootstrap ranges for 'share explained' and n0.
#     (c) Within-position agreement of tracked and official speed.   Outputs: ablation.csv, n0_bootstrap.csv, sensor_agreement_by_position.csv
import sys, io, contextlib, numpy as np, pandas as pd
D, O = sys.argv[1], sys.argv[2]
SRC = globals().get("SRC12") or open(__import__("os").path.join(__import__("os").path.dirname(__import__("os").path.abspath(__file__)), "12_snaps_equivalent.py")).read()   # SRC12 is set by the notebook
SRC = SRC.replace('pd.DataFrame(out).to_csv(f"{O}/snaps_equivalent_results.csv",index=False); print("\\nsaved")', 'pass')
FULL = ["forty", "ten_yd_split", "combine_weight"]
SRC = SRC.replace('T = cr[BASE+["combine_position"]]', 'T = cr[["forty","ten_yd_split","combine_weight","combine_position"]]')
SRC = SRC.replace("dropna(subset=BASE+FEAT)", 'dropna(subset=["forty","ten_yd_split","combine_weight"]+FEAT)')   # keep identical players across arms
def run(base):
    s = SRC.replace('BASE=["forty","ten_yd_split","combine_weight"]', f"BASE={base!r}")
    ns = {"__name__": "__main__"}
    sys.argv = ["x", D, O]
    with contextlib.redirect_stdout(io.StringIO()): exec(compile(s, "12_patched", "exec"), ns)
    return pd.DataFrame(ns["out"])
rng = np.random.default_rng(2027)
arms = {"full (40, 10-yd, weight)": FULL, "weight only": ["combine_weight"], "40 time + 10-yd split only": ["forty", "ten_yd_split"]}
rows = []
for lab, base in arms.items():
    r = run(base)
    r = r[r.k.isin([0, 25])][["outcome", "k", "mse_pos", "mse_comb", "gain_pos_comb", "lo", "hi", "share_explained", "n0_comb"]]; r.insert(0, "prior", lab); rows.append(r)
A = pd.concat(rows); A.to_csv(f"{O}/ablation.csv", index=False)
pd.set_option("display.width", 250)
P = A.pivot_table(index=["outcome", "k"], columns="prior", values="mse_comb"); P["position mean"] = A[A.prior.str.startswith("full")].set_index(["outcome", "k"]).mse_pos
print("MSE vs last-100 mean (lower is better):"); print(P.round(4).to_string())
S = A.pivot_table(index="outcome", columns="prior", values="share_explained"); print("\nshare of true between-player variance explained:"); print((S * 100).round(0).to_string())
# (b) bootstrap of share explained and n0 (player bootstrap over the pool, full prior)
import importlib
ns = {}
src = (globals().get("SRC12") or open(__import__("os").path.join(__import__("os").path.dirname(__import__("os").path.abspath(__file__)), "12_snaps_equivalent.py")).read()).split("out=[]")[0]   # setup + helper functions only
sys.argv = ["x", D, O]
with contextlib.redirect_stdout(io.StringIO()): exec(compile(src, "12_setup", "exec"), ns)
T, tabs, fit_prior, BASE_, FEAT_ = ns["T"], ns["tabs"], ns["fit_prior"], ns["BASE"], ns["FEAT"]
brow = []
for name, (Pos, df) in tabs.items():
    df = df.sort_values(["nfl_id", "game_id", "play_id"]); ok = T[(T.combine_position == Pos)].dropna(subset=BASE_ + FEAT_).index; df = df[df.nfl_id.isin(ok)]
    g = df.groupby("nfl_id").y; st = pd.DataFrame({"n": g.size(), "mean": g.mean(), "var": g.var(ddof=1)}); pool = st[st.n >= 30].index.to_numpy()
    if (st.n >= 200).sum() < 15: continue
    sh, n0 = [], []
    for b in range(500):
        ids = pool[rng.integers(0, len(pool), len(pool))]
        # resampled players share one row of T/stats; fit_prior only needs T/stats rows by id, duplicates are fine for .loc
        s2 = ((st.loc[ids, "n"] - 1) * st.loc[ids, "var"]).sum() / (st.loc[ids, "n"] - 1).sum()
        tp = fit_prior(list(ids), st, [], s2)[3]; tb = max(fit_prior(list(ids), st, BASE_, s2)[3], 0.02 * tp)
        sh.append(1 - tb / tp); n0.append(s2 / tb)
    s2 = ((st.loc[pool, "n"] - 1) * st.loc[pool, "var"]).sum() / (st.loc[pool, "n"] - 1).sum()
    brow.append(dict(outcome=name, share_lo=np.percentile(sh, 2.5), share_hi=np.percentile(sh, 97.5), n0_lo=np.percentile(n0, 2.5), n0_hi=np.percentile(n0, 97.5)))
B = pd.DataFrame(brow); B.to_csv(f"{O}/n0_bootstrap.csv", index=False); print("\nplayer-bootstrap 95% ranges (pool resampling):"); print(B.round(2).to_string(index=False))
# (c) sensor vs stopwatch agreement within position
cr = pd.read_csv(f"{D}/combine_results.csv", na_values=["NA"]).set_index("nfl_id"); f = pd.read_csv(f"{O}/forty_run_features.csv").groupby("nfl_id")[["vmax", "v10"]].mean()
X = cr[["forty", "ten_yd_split", "combine_position"]].join(f).dropna(); rows = []
for pos, g in list(X.groupby("combine_position")) + [("ALL", X)]:
    rows.append(dict(position=pos, n=len(g), r_vmax_vs_forty=g.vmax.corr(g.forty), r_v10_vs_ten_yd_split=g.v10.corr(g.ten_yd_split)))
C = pd.DataFrame(rows); C.to_csv(f"{O}/sensor_agreement_by_position.csv", index=False); print("\ntracked vs official, by position:"); print(C.round(2).to_string(index=False))

Sensitivity: the WR separation adjustment (route, coverage, alignment, cushion) refit without each held-out player.

In [ ]:
import sys
sys.argv = ["x", DATA, OUTDIR]
# --- 18_wr_adjustment_holdout.py ---
# WR adjusted separation with the route/coverage/alignment/cushion adjustment refit WITHOUT each held-out player (closes the
# leakage noted in review: in 12_snaps_equivalent.py the adjustment is fit once on all WR plays).  Compares to the original errors.
import sys, io, contextlib, numpy as np, pandas as pd
D, O = sys.argv[1], sys.argv[2]
SRC = globals().get("SRC12") or open(__import__("os").path.join(__import__("os").path.dirname(__import__("os").path.abspath(__file__)), "12_snaps_equivalent.py")).read()
ns = {}; sys.argv = ["x", D, O]
with contextlib.redirect_stdout(io.StringIO()): exec(compile(SRC.split("out=[]")[0], "12_setup", "exec"), ns)
T, tabs, fit_prior, predict, BASE, FEAT, KS, LATE = (ns[k] for k in ("T", "tabs", "fit_prior", "predict", "BASE", "FEAT", "KS", "LATE"))
P, w = tabs["WR adjusted separation"]
w = w.sort_values(["nfl_id", "game_id", "play_id"]).reset_index(drop=True)
Xd = pd.get_dummies(w[["route_ran", "team_coverage_man_zone", "lined_up_position"]].fillna("NA"), drop_first=True).astype(float); Xd["cushion"] = w.cushion.values
A = np.c_[np.ones(len(Xd)), Xd.values]; sep = w.separation_at_pass_forward.values; ids_all = w.nfl_id.values
ok = T[(T.combine_position == P)].dropna(subset=BASE + FEAT).index
keep = np.isin(ids_all, ok)
st0 = w[keep].groupby("nfl_id").y.agg(n="size"); pool = st0[st0.n >= 30].index.tolist(); test = st0[st0.n >= KS[-1] + LATE].index.tolist()
rng = np.random.default_rng(2027); E = {k: {"pos": [], "base": []} for k in KS}
for pid in test:
    beta = np.linalg.lstsq(A[ids_all != pid], sep[ids_all != pid], rcond=None)[0]
    y = sep - A @ beta; df = pd.DataFrame({"nfl_id": ids_all, "y": y})[keep]
    g = df.groupby("nfl_id").y; st = pd.DataFrame({"n": g.size(), "mean": g.mean(), "var": g.var(ddof=1)})
    rest = [i for i in pool if i != pid]; rs = st.loc[rest]; sig2 = ((rs.n - 1) * rs["var"]).sum() / (rs.n - 1).sum()
    pri = {}
    for arm, cols in (("pos", []), ("base", BASE)):
        b, mu, sd, t2 = fit_prior(rest, st, cols, sig2)
        if arm != "pos": t2 = max(t2, 0.02 * fit_prior(rest, st, [], sig2)[3])
        pri[arm] = (predict(b, mu, sd, pid, cols), t2)
    yy = df[df.nfl_id == pid].y.values; late = yy[-LATE:].mean()
    for k in KS:
        for arm in pri:
            mu0, t2 = pri[arm]; post = mu0 if k == 0 else (k / sig2 * yy[:k].mean() + mu0 / t2) / (k / sig2 + 1 / t2)
            E[k][arm].append((post - late) ** 2)
orig = pd.read_csv(f"{O}/snaps_equivalent_results.csv"); orig = orig[orig.outcome == "WR adjusted separation"].set_index("k")
rows = []
for k in KS:
    a, b = np.array(E[k]["pos"]), np.array(E[k]["base"]); d = a - b; n = len(d)
    bs = [d[rng.integers(0, n, n)].mean() for _ in range(2000)]
    rows.append(dict(k=k, n_test=n, mse_pos_holdout=a.mean(), mse_comb_holdout=b.mean(), gain_holdout=d.mean(), lo_holdout=np.percentile(bs, 2.5), hi_holdout=np.percentile(bs, 97.5),
                     mse_pos_orig=orig.loc[k, "mse_pos"], mse_comb_orig=orig.loc[k, "mse_comb"], gain_orig=orig.loc[k, "gain_pos_comb"], lo_orig=orig.loc[k, "lo"], hi_orig=orig.loc[k, "hi"]))
R = pd.DataFrame(rows); R.to_csv(f"{O}/wr_adjustment_holdout.csv", index=False)
pd.set_option("display.width", 250); print("WR separation: adjustment refit without the held-out player vs original (adjustment fit on all WR plays)"); print(R.round(4).to_string(index=False))

## 10. Figures

In [ ]:
import sys
sys.argv = ["x", DATA, OUTDIR]
# --- 14_figures.py ---
import sys, os, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
D,O=sys.argv[1],sys.argv[2]; os.makedirs(f"{O}/figures",exist_ok=True)
BLUE,ORANGE,AQUA="#257de0","#f47230","#00b894"; INK="#191919"; INK2="#666666"; GRID="#e7e7e7"; AXIS="#c2c2c2"
plt.rcParams.update({"font.family":["Lato","DejaVu Sans"],"font.size":10,"axes.edgecolor":AXIS,"axes.labelcolor":INK2,"xtick.color":INK2,"ytick.color":INK2,
 "axes.spines.top":False,"axes.spines.right":False,"axes.grid":True,"grid.color":GRID,"grid.linewidth":1,"axes.axisbelow":True,"figure.facecolor":"white","axes.facecolor":"white","text.color":INK})
def legend(fig,items,loc="lower center",ncol=3,y=-0.01):
    h=[Line2D([0],[0],marker="o",color="none",markerfacecolor=c,markeredgecolor="white",markeredgewidth=1.5,markersize=9,label=l) for l,c in items]
    fig.legend(handles=h,loc=loc,ncol=ncol,frameon=False,bbox_to_anchor=(0.5,y),labelcolor=INK2,fontsize=10,handletextpad=0.3,columnspacing=1.6)
# ---------- Fig 1: reliability vs redundancy of 40-yard-dash tracking features
cr=pd.read_csv(f"{D}/combine_results.csv",na_values=["NA"]).set_index("nfl_id"); f=pd.read_csv(f"{O}/forty_run_features.csv")
names={"v10":"Speed at 10 yd","v20":"Speed at 20 yd","v30":"Speed at 30 yd","vmax":"Top speed","t40":"Time to 38 yd","t10":"Time to 10 yd","peak_acc":"Peak acceleration","d95":"Yards to 95% of top speed","t_vmax":"Time of top speed","maint":"Speed maintenance"}
speed={"v10","v20","v30","vmax","t40"}
two=f[f.run<=2].pivot(index="nfl_id",columns="run",values=list(names)).dropna()
P=f.groupby("nfl_id")[list(names)].mean().join(cr[["forty","ten_yd_split","combine_weight"]]).dropna()
X=np.c_[np.ones(len(P)),P.forty,P.ten_yd_split,P.combine_weight]; pts={}
for m in names:
    y=P[m].values; b=np.linalg.lstsq(X,y,rcond=None)[0]; r2=1-((y-X@b)**2).sum()/((y-y.mean())**2).sum()
    pts[m]=(r2,np.corrcoef(two[(m,1)],two[(m,2)])[0,1])
fig,ax=plt.subplots(figsize=(7.4,5.2))
for m,(x,y) in pts.items(): ax.plot(x,y,"o",color=BLUE if m in speed else ORANGE,markersize=9,markeredgecolor="white",markeredgewidth=2,zorder=3)
off={"v10":(-9,-12,"right"),"v20":(-12,-3,"right"),"v30":None,"vmax":None,"t40":(9,-4,"left"),"t10":(9,-4,"left"),"peak_acc":(9,-4,"left"),"d95":(0,10,"center"),"t_vmax":(9,-10,"left"),"maint":(-9,6,"right")}
for m,o in off.items():
    if o is None: continue
    lab="Speed at 20 and 30 yd,\ntop speed" if m=="v20" else names[m]
    ax.annotate(lab,pts[m],xytext=(o[0],o[1]),textcoords="offset points",ha=o[2],fontsize=9.5,color=INK2)
ax.set_xlim(0,1.04); ax.set_ylim(0,1.05); ax.set_xlabel("Share already explained by official 40 time, 10-yd split and weight (R²)"); ax.set_ylabel("Test-retest reliability (run 1 vs run 2, r)")
ax.text(0.02,1.0,"Reliable and not redundant would sit here: no feature does",fontsize=9.5,color=INK2,style="italic",va="top")
for s in ("left","bottom"): ax.spines[s].set_color(AXIS)
fig.suptitle("Figure 1. Tracked 40-yard-dash speed repeats well but repeats the stopwatch",x=0.02,ha="left",fontsize=12,fontweight="semibold",y=0.985)
legend(fig,[("Overall speed and time",BLUE),("Acceleration shape and split timing",ORANGE)],ncol=2,y=0.0); fig.tight_layout(rect=(0,0.05,1,0.95)); fig.savefig(f"{O}/figures/fig1_reliability_vs_redundancy.png",dpi=200); plt.close(fig)
# ---------- Fig 2: error vs snaps
R=pd.read_csv(f"{O}/snaps_equivalent_results.csv")
panels=[("WR adjusted separation","WR separation (yd²)"),("DL get-off (lower=faster)","DL get-off (s²)"),("DL in-game top speed","DL in-game top speed ((yd/s)²)")]
fig,axs=plt.subplots(1,3,figsize=(11,4.1))
for ax,(key,title) in zip(axs,panels):
    d=R[R.outcome==key].sort_values("k")
    for col,c in (("mse_raw",AQUA),("mse_pos",ORANGE),("mse_comb",BLUE)):
        ax.plot(d.k,d[col],color=c,linewidth=2,marker="o",markersize=8,markeredgecolor="white",markeredgewidth=2,solid_capstyle="round",zorder=3)
    ax.set_yscale("log"); ax.set_xticks([0,10,25,50,100]); ax.set_title(title,loc="left",fontsize=10.5,color=INK,fontweight="semibold"); ax.set_xlabel("NFL snaps used (first k plays)")
    for s in ("left","bottom"): ax.spines[s].set_color(AXIS)
axs[0].set_ylabel("Squared error vs last-100-play average (log scale)")
fig.suptitle("Figure 2. Most of the gain over raw early averages comes from shrinking toward a prior",x=0.01,ha="left",fontsize=12,fontweight="semibold",y=0.99)
legend(fig,[("Average of early snaps only",AQUA),("Shrunk to position average",ORANGE),("Shrunk to Combine prior",BLUE)],ncol=3,y=0.0); fig.tight_layout(rect=(0,0.07,1,0.93)); fig.savefig(f"{O}/figures/fig2_error_vs_snaps.png",dpi=200); plt.close(fig)
# ---------- Fig 3: prior-weight dumbbell
S=R.drop_duplicates("outcome").copy(); lab={"DL get-off (lower=faster)":"DL get-off","WR adjusted separation":"WR separation","OL pressure allowed":"OL pressure allowed","WR in-game top speed":"WR top speed","DL in-game top speed":"DL top speed","DB in-game top speed":"DB top speed","OL in-game top speed":"OL top speed"}
B3=pd.read_csv(f"{O}/n0_bootstrap.csv").set_index("outcome"); S["lab"]=S.outcome.map(lab); S=S.sort_values("n0_comb").reset_index(drop=True)
H=pd.read_csv(f"{O}/robustness_holm.csv").set_index("outcome")
def verdict(o):
    h=H.loc[o]
    if h.rule_met_in_seeds_of_5>=5: return "rule met; survives Holm" if h["holm_reject_at_.05"] else "rule met; fails Holm"
    return "rule not met"
fig,ax=plt.subplots(figsize=(8.4,4.5)); ax.grid(axis="y",visible=False)
for i,r in S.iterrows():
    ax.plot([B3.loc[r.outcome,"n0_lo"],B3.loc[r.outcome,"n0_hi"]],[i,i],color=BLUE,linewidth=1.5,alpha=0.45,zorder=1,solid_capstyle="butt")
    ax.plot([r.n0_pos,r.n0_comb],[i,i],color=AXIS,linewidth=2,zorder=2,solid_capstyle="round")
    ax.plot(r.n0_pos,i,"o",color=ORANGE,markersize=11,markeredgecolor="white",markeredgewidth=2,zorder=3)
    ax.plot(r.n0_comb,i,"o",color=BLUE,markersize=8,markeredgecolor="white",markeredgewidth=2,zorder=4)
    ax.text(1000,i,verdict(r.outcome),va="center",fontsize=9.5,color=INK2,clip_on=False)
ax.set_xscale("log"); ax.set_xlim(5,700); ax.set_xticks([10,30,100,300]); ax.set_xticklabels(["10","30","100","300"]); ax.set_yticks(range(len(S))); ax.set_yticklabels(S.lab); ax.set_xlabel("Model-implied prior weight, in independent-play equivalents (log scale)\nBlue band: 95% player-bootstrap range for the Combine-informed weight")
for s in ("left","bottom"): ax.spines[s].set_color(AXIS)
fig.suptitle("Figure 3. Model-implied weight of a Combine-informed prior",x=0.02,ha="left",fontsize=12,fontweight="semibold",y=0.985)
legend(fig,[("Prior = position average",ORANGE),("Prior = 40 time, 10-yd split, weight",BLUE)],ncol=2,y=0.0); fig.tight_layout(rect=(0,0.06,0.78,0.95)); fig.savefig(f"{O}/figures/fig3_snaps_equivalent.png",dpi=200); plt.close(fig)
# ---------- Fig 4: sensor vs stopwatch, paired held-out error difference (forest plot)
Q=pd.read_csv(f"{O}/sensor_vs_stopwatch.csv"); Q=Q[Q.k.isin([0,25])].copy()
Q["rel"]=Q.gain/Q.mse_pos; Q["rlo"]=Q.lo/Q.mse_pos; Q["rhi"]=Q.hi/Q.mse_pos
order=["DL get-off","WR adj separation","WR top speed","DL top speed","DB top speed","OL top speed"]
Q["o"]=Q.outcome.str.replace(" (lower=faster)","",regex=False); Q["o"]=Q.o.replace({"WR adj separation":"WR separation"}); order=["DL get-off","WR separation","WR top speed","DL top speed","DB top speed","OL top speed"]
fig,ax=plt.subplots(figsize=(8.2,4.4)); ax.grid(axis="y",visible=False)
for j,(kk,col,dy) in enumerate(((0,BLUE,0.14),(25,ORANGE,-0.14))):
    for i,o in enumerate(order[::-1]):
        r=Q[(Q.o==o)&(Q.k==kk)]
        if r.empty: continue
        r=r.iloc[0]; ax.plot([r.rlo,r.rhi],[i+dy,i+dy],color=col,linewidth=2,zorder=3,solid_capstyle="butt"); ax.plot(r.rel,i+dy,"o",color=col,markersize=8,markeredgecolor="white",markeredgewidth=2,zorder=4)
ax.axvline(0,color=INK2,linewidth=1,zorder=2)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order[::-1]); ax.set_ylim(-0.6,len(order)-0.4)
ax.set_xlabel("Stopwatch-prior error minus sensor-prior error, as a share of position-average error (95% CI)\nRight of zero: sensor prior better; left: stopwatch prior better")
for s_ in ("left","bottom"): ax.spines[s_].set_color(AXIS)
fig.suptitle("Figure 4. No detectable difference between sensor and stopwatch priors",x=0.02,ha="left",fontsize=12,fontweight="semibold",y=0.985)
legend(fig,[("k = 0 early plays",BLUE),("k = 25 early plays",ORANGE)],ncol=2,y=0.0); fig.tight_layout(rect=(0,0.06,1,0.95)); fig.savefig(f"{O}/figures/fig4_sensor_vs_stopwatch.png",dpi=200); plt.close(fig)
print("figures:",sorted(os.listdir(f"{O}/figures")))

In [ ]:
import shutil
from IPython.display import Image, display
RESULTS = "/kaggle/working/results" if ON_KAGGLE else OUTDIR + "/results"
os.makedirs(RESULTS, exist_ok=True)
for f in ("translation_results.csv", "snaps_equivalent_results.csv", "sensor_vs_stopwatch.csv", "robustness_holm.csv", "ablation.csv", "n0_bootstrap.csv", "sensor_agreement_by_position.csv", "influence.csv", "wr_adjustment_holdout.csv"):   # aggregate tables only
    shutil.copy(f"{OUTDIR}/{f}", RESULTS)
for f in sorted(os.listdir(f"{OUTDIR}/figures")):
    shutil.copy(f"{OUTDIR}/figures/{f}", RESULTS); display(Image(f"{RESULTS}/{f}"))
print("published to", RESULTS, ":", sorted(os.listdir(RESULTS)))